# Async basics (Part 2)

LLM calls are **I/O-bound**: most of the time is spent waiting for the network. `async`/`await` lets us wait for many calls at once instead of one after another.

> **Python version:** `async`/`await` syntax is available from **Python 3.5** (PEP 492). `asyncio.run()` — the usual entry point in scripts — was added in **3.7**. This repo requires **3.13+** (`pyproject.toml`).

### Three things to know

| | Concept | In one line |
|---|---|---|
| 📝 | `async def` | Defines a coroutine; calling it returns an awaitable (it does not run yet). |
| ⏳ | `await` | Runs it and yields control while waiting. |
| 🚀 | `asyncio.gather(...)` | Runs many coroutines concurrently and returns all results. |
| 🚦 | `asyncio.Semaphore(5)` | Caps how many coroutines run at once — a gate you choose. |

---

### 🍽️ Food metaphor

You are at a restaurant with a **busy kitchen** — several cooks can work at the same time 👨‍🍳👨‍🍳👨‍🍳. You want to order food for a few friends, each on a **separate cheque** 🧾.

| Step | Code | What happens |
|:---:|---|---|
| 1️⃣ | `async def get_food` | 📝 You decide what each dish should be (the recipe for one friend's order). |
| 2️⃣ | `get_food("Alice")` | 🧾 You fill out **Alice's cheque**, but nothing is cooking yet — you are just holding the paper. |
| 3️⃣ | `await get_food("Alice")` | 🍳 You hand Alice's cheque to the kitchen. While her dish cooks, you can hand in Bob's cheque, then Carol's. Different cooks handle different orders. |
| 4️⃣ | `asyncio.gather(get_food("Alice"), get_food("Bob"), get_food("Carol"))` | ⚡ You submit **all cheques at once**. Three cooks work in parallel. Five 1-minute dishes take ~1 minute, not ~5. |

---

### 🔥 Semaphore metaphor — only five stoves

The kitchen still has several cooks, but only **5 stoves** 🔥🔥🔥🔥🔥. You have **10 friends** and **10 cheques** 🧾 — you cannot cook all ten meals at once.

| Step | Code | What happens |
|:---:|---|---|
| 1️⃣ | `asyncio.Semaphore(5)` | 🚦 The manager says: "At most **5 dishes on the stoves** at any moment." |
| 2️⃣ | `async with sem:` before `await get_food(...)` | 🔒 A cook **claims a stove** before starting. If all 5 are busy, this order waits in line. |
| 3️⃣ | `await get_food("Alice")` | 🍳 The dish cooks on that stove (~1 min). When done, the stove is free for the next cheque. |
| 4️⃣ | `asyncio.gather(...)` × 10 | ⚡ You submit **all 10 cheques**, but only **5 cook at once**. Ten 1-minute dishes take **~2 minutes**, not ~1 (unlimited stoves) and not ~10 (one stove). |

---

We use this in the model-stability test: fire the same request 5 times concurrently and check every reply agrees.

In [1]:
import asyncio
import time


async def fake_llm_call(i: int) -> str:
    await asyncio.sleep(1)  # pretend this is a 1s network round-trip
    return f"reply-{i}"


async def run_sequential() -> list[str]:
    return [await fake_llm_call(i) for i in range(5)]


async def run_concurrent() -> list[str]:
    return await asyncio.gather(*(fake_llm_call(i) for i in range(5)))

## How Async Works under the Hood 🧠

To understand why `async`/`await` is so powerful, let's look at how it works in the background, how it differs from threads/processes, and why it is perfect for LLM-based applications.

### 1. The Analogy: The Single-Waiter Restaurant 🍽️

Imagine a restaurant with one waiter (our **Event Loop**):
* **Synchronous (The Old/Slow way)**: The waiter takes Table 1's order, walks to the kitchen, and **stands there doing nothing** while the chef cooks. Only after the meal is ready does the waiter serve Table 1 and move to Table 2. If cooking takes 10 minutes, Table 2 waits at least 10 minutes just to order!
* **Asynchronous (Async/Await)**: The waiter takes Table 1's order, hands it to the kitchen (`await`), and **immediately** walks over to Table 2 to take their order. While Table 1's food is cooking (network I/O), the waiter keeps serving others. When Table 1's food is ready, the kitchen calls the waiter, who delivers it at the next opportunity.

In Python, the **Event Loop** is the waiter. The network requests to the LLM are the kitchen cooking the food.

---

### 2. How many processes/threads are spun up? 🧵
**Exactly ONE process and ONE thread!** 

Unlike other concurrency models:
* No new OS threads are spawned.
* No new processes are created.
* Everything runs on the single, main thread of your Python program.
* It achieves concurrency not by running code at the exact same millisecond, but by **never standing idle** while waiting for external things (like network responses).

---

### 3. Why not use old-school threads (`threading`)? 🚫

You might ask: *Why not just spin up a thread for each API call?* Here is why Python developers prefer `asyncio` for I/O-bound tasks:

1. **The GIL (Global Interpreter Lock)**: Python has a lock that ensures only one thread can execute Python code at a time. While threads can run concurrently when waiting for I/O, Python threads don't give you true CPU parallelism.
2. **Heavy Memory Overhead**: Each OS thread has a fixed memory overhead (usually 8MB stack size by default). Spinning up 100 threads can quickly eat up hundreds of megabytes of RAM. Async coroutines are lightweight Python objects, taking only a few *kilobytes* each. You can run 10,000+ async tasks on a single laptop without breaking a sweat!
3. **Context Switching Costs**: The operating system constantly pauses and resumes threads (preemptive multitasking). This "context switching" requires CPU cycles. Async uses **cooperative multitasking**, meaning tasks voluntarily yield control (using `await`) only when they have nothing to do, which is extremely efficient.
4. **Race Conditions & Safety**: In a threaded program, the OS can interrupt a thread at any line of code. If two threads modify the same variable, you get race conditions and bugs. With `asyncio`, code only pauses at explicit `await` keywords. You always know exactly where your code can pause, making it thread-safe by design and preventing hard-to-debug race conditions.

## Sequential vs concurrent

Five 1-second calls take ~5s sequentially but ~1s concurrently. In a notebook there is already a running event loop, so we `await` directly. In a plain script you would call `asyncio.run(run_concurrent())`.

In [2]:
start = time.perf_counter()
seq = await run_sequential()
print(f"sequential: {seq} in {time.perf_counter() - start:.1f}s")

start = time.perf_counter()
conc = await run_concurrent()
print(f"concurrent: {conc} in {time.perf_counter() - start:.1f}s")

sequential: ['reply-0', 'reply-1', 'reply-2', 'reply-3', 'reply-4'] in 5.1s


concurrent: ['reply-0', 'reply-1', 'reply-2', 'reply-3', 'reply-4'] in 1.0s


## How this maps to the project

`LLMClient` exposes `acomplete` / `astructured` (built on LangChain's `ainvoke`). The model-stability test in `tests/integration/test_model_stability.py` fans out 5 concurrent `astructured` calls with `asyncio.gather` and asserts they all return the same intent - the determinism check at temperature 0.

When you need to cap load (many LLM calls, rate limits), wrap `acomplete` with `async with sem:` the same way as `call_limited` above.